## Setup and Imports

In [ ]:
# Standard Libriaries

import sys
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')  #Suppress minor warnings for cleaner output

# Custom modules
project_root = Path().resolve().parents[1] 
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Data manipulation
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Custom helper functions
from utils.helpers import load_orders, set_style, save_figure

set_style()

print('All imports successful✓')
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')



In [ ]:
# Load the dataset

df_raw = load_orders(cleaned=False) # never modified
df = df_raw.copy()                  # working copy - safe to transform

print('Dataset Overview')
print(f'Rows:  {len(df):,}')
print(f'Columns:  {len(df.columns):,}')
print(f'\nColumn names:')
for col in df.columns:
    print(f'  {col}')

In [ ]:
df.head()

In [ ]:
print('Data Types and Null Counts')
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
print(info_df.to_string())

In [ ]:
# Parse the timestamp columns into proper datetime objects
# errors='coerce' means if a value cannot be parsed as a date,
# it becomes NaT (Not a Time) rather than crashing the whole operation

df['PURCHASE_TS'] = pd.to_datetime(df['PURCHASE_TS'], errors='coerce')
df['SHIP_TS'] = pd.to_datetime(df['SHIP_TS'], errors='coerce')

# Calculate fulfilment days: ship date minus purchase date
# Positive = normal (shipped after purchase)
# Zero     = same-day dispatch
# Negative = ANOMALY (shipped before purchase — impossible in reality)
df['FULFILMENT_DAYS'] = (df['SHIP_TS'] - df['PURCHASE_TS']).dt.days

print('Timestamps parsed successfully ✓')
print(f"Purchase TS range: {df['PURCHASE_TS'].min()} → {df['PURCHASE_TS'].max()}")
print(f"Ship TS range:     {df['SHIP_TS'].min()} → {df['SHIP_TS'].max()}")

## Quantify Anomalies

In [ ]:
# Segment the data based on fulfilment days
# This gives clear picture of full distribution and highlights any anomalies or missing data

normal = df[df['FULFILMENT_DAYS'] > 0] # normal orders that shipped after purchase
same_day = df[df['FULFILMENT_DAYS'] == 0] # same-day orders that shipped on the purchase day
anomalies = df[df['FULFILMENT_DAYS'] < 0] # anomalies that shipped before purchase (impossible, likely data errors)
missing = df[df['FULFILMENT_DAYS'].isna()] # missing timestamps where either PURCHASE_TS or SHIP_TS is null, making fulfilment days impossible to calculate

total = len(df)

print('Fulfilment Days Segmentation')
print(f"{'Segment':<30} {'Count':>8} {'% of Total':>12}")
print('-' * 52)
print(f"{'Normal (shipped after purchase)':<30} {len(normal):>8,} {len(normal)/total*100:>11.2f}%")
print(f"{'Same Day (shipped on purchase day)':<30} {len(same_day):>8,} {len(same_day)/total*100:>11.2f}%")
print(f"{'Anomalies (shipped BEFORE purchase)':<30} {len(anomalies):>8,} {len(anomalies)/total*100:>11.2f}%")
print(f"{'Missing timestamps':<30} {len(missing):>8,} {len(missing)/total*100:>11.2f}%")
print('-' * 52)
print(f"{'Total':<30} {total:>8,} {'100.0%':>12}")

In [ ]:
# Analyze the anomalous records in more detail to understand the severity of the data issues

print('Anomalous Records - Fulfilment Days Statistics')
stats = anomalies['FULFILMENT_DAYS'].describe()
print(f" Most extreme:   {stats['min']:.0f} days (shipped {abs(stats['min']):.0f} days BEFORE purchase)")
print(f" Median:         {stats['50%']:.0f} days")
print(f" Mean:           {stats['mean']:.1f} days")
print(f" 75th pct:       {stats['75%']:.0f} days")
print()

# Distribution of anomaly severity
# - how many records are just slightly negative (e.g., -1 day) vs. how many are extreme (e.g., -30 days or more)

print(' Distribution of Anomaly Severity')
bins = [
    ('1 day before purchase', anomalies[anomalies['FULFILMENT_DAYS'] == -1]),  # shipped 1 day before purchase
    ('2-7 days before purchase', anomalies[(anomalies['FULFILMENT_DAYS'] >= -7) & (anomalies['FULFILMENT_DAYS'] < -1)]),  # shipped 2 to 7 days before purchase
    ('1-4 weeks before purchase', anomalies[(anomalies['FULFILMENT_DAYS'] >= -30) & (anomalies['FULFILMENT_DAYS'] < -7)]),  # shipped 8 to 30 days before purchase
    ('1-3 months before purchase', anomalies[(anomalies['FULFILMENT_DAYS'] >= -90) & (anomalies['FULFILMENT_DAYS'] < -30)]),  # shipped 31 to 90 days before purchase
    ('Over 3 months before purchase', anomalies[anomalies['FULFILMENT_DAYS'] < -90])  # shipped more than 90 days before purchase
]
for label, subset in bins:
    print(f"  {label:<25} {len(subset):>6,} records  ({len(subset)/len(anomalies)*100:.1f}%)")

## Pattern Investigation

In [ ]:
# Flag each record for comparison between anomalies and normal orders, to facilitate further analysis and visualisation
# Binary flag column to indicate whether a record is anomalous or not, based on the fulfilment days

df['IS_ANOMALY'] = df['FULFILMENT_DAYS'] < 0

# Pattern 1: By Purchase platform
# Are anomalies more common on website vs mobile app? This could indicate platform-specific data issues or user behavior differences.
# If mobile app has far more anomalies, 
# it might suggest a bug in the app's timestamp handling or a user behavior pattern that leads to more data issues. 
# Conversely, if the website has more anomalies, it could point to issues in the web order processing system.

print('Anomalies by Purchase Platform')
platform_analysis = df.groupby('PURCHASE_PLATFORM').agg(
    total_orders=('ORDER_ID', 'count'),
    anomalies_count=('IS_ANOMALY', 'sum')
).assign(
    anomaly_rate = lambda x: (x['anomalies_count'] / x['total_orders'] * 100).round(1)
).sort_values('anomaly_rate', ascending=False)
print(platform_analysis.to_string())

In [ ]:
# Pattern 2: By Product
# Are certain products more prone to anomalies?
# If a specific product (e.g., "iPhone 12") has a much higher anomaly rate than others, 
# it could indicate product-specific data issues, such as a particular supplier's data feed being
# more error-prone, or it could reflect real-world differences in how certain products are processed and shipped
# or it might suggest a data entry issue with that SKU

print('Anomalies by Product')
product_analysis = df.groupby('PRODUCT_NAME').agg(
    total_orders=('ORDER_ID', 'count'),
    anomalies_count=('IS_ANOMALY', 'sum')
).assign(
    anomaly_rate = lambda x: (x['anomalies_count'] / x['total_orders'] * 100).round(1)
).sort_values('anomaly_rate', ascending=False)
print(product_analysis.to_string())

In [ ]:
# Pattern 3: By top 15 countries
# Geographic concentration may indicate a timezone issue
# e.g. a country far from UTC might have timestamps recorded in local time 
# but interpreted as UTC, leading to systematic negative fulfilment days for orders from that country
print('Anomalies by Country (Top 15 based on order volume)')
country_analysis = df.groupby('COUNTRY_CODE').agg(
    total_orders=('ORDER_ID', 'count'),
    anomalies_count=('IS_ANOMALY', 'sum')
).assign(
    anomaly_rate = lambda x: (x['anomalies_count'] / x['total_orders'] * 100).round(1)
).sort_values('total_orders', ascending=False).head(15)
print(country_analysis.to_string())

In [ ]:
# Pattern 4: By month/year
# Did anomalies rate change over time? 
# A sudden spike in anomalies during a specific month or year could indicate a data processing issue that started at that time, 
# such as a software update that introduced a bug in timestamp handling, 
# or a change in the order processing system that affected how timestamps are recorded.

print('Anomalies by Month/Year')
df['YEAR_MONTH'] = df['PURCHASE_TS'].dt.to_period('M')  # extract year and month from purchase timestamp

time_analysis = df.groupby('YEAR_MONTH').agg(
    total_orders=('ORDER_ID', 'count'),
    anomalies_count=('IS_ANOMALY', 'sum')
).assign(
    anomaly_rate = lambda x: (x['anomalies_count'] / x['total_orders'] * 100).round(1)
)
print(time_analysis.to_string())

In [ ]:
# Pattern 5: Marketing channel
# Are anomalies more common in orders that came through certain marketing channels?

print('Anomalies by Marketing Channel')
channel_analysis = df.groupby('MARKETING_CHANNEL').agg(
    total_orders=('ORDER_ID', 'count'),
    anomalies_count=('IS_ANOMALY', 'sum')
).assign(
    anomaly_rate = lambda x: (x['anomalies_count'] / x['total_orders'] * 100).round(1)
).sort_values('anomaly_rate', ascending=False)
print(channel_analysis.to_string())

## Visualisations

In [ ]:
# Chart 1 Overall segmentation
# Show scale of problem at a glance - how many orders are normal vs same-day vs anomalies vs missing timestamps


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

segments = ['Normal\n(shipped after)', 'Same day', 'Anomalies\n(shipped before)', 'Missing\ntimestamps']
counts = [len(normal), len(same_day), len(anomalies), len(missing)]
colours = ['#2E75B6', '#70AD47', '#C00000', '#808080']

bars = axes[0].bar(segments, counts, color=colours, width=0.5)
axes[0].set_title('Order Fulfilment Segmentation', fontsize=14, fontweight='medium')
axes[0].set_ylabel('Number of Orders', fontsize=12)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))  # format y-axis with commas

# Add count labels on top of bars
for bar, count in zip(bars, counts):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, 
        bar.get_height() + 100,  # position label slightly above the bar
        f'{count:,}', 
        ha='center', va='bottom', fontsize=10
    )
    
# Horizontal bar showing percentage breakdown of the segments, to visually represent the proportion of anomalies compared to normal orders and same-day orders
percentages = [c / total * 100 for c in counts]

# Build the bar left to right
left = 0
for pct, colour, label in zip(percentages, colours, segments):
    axes[1].barh(
        0, pct, left=left,
        color=colour, height=0.5,
        label=f'{label.replace(chr(10), " ")} ({pct:.1f}%)'
    )
    # Only add a label if the segment is wide enough to read
    if pct > 1:
        axes[1].text(
            left + pct / 2, 0,
            f'{pct:.1f}%',
            ha='center', va='center',
            fontsize=10, color='white', fontweight='medium'
        )
    left += pct

axes[1].set_xlim(0, 100)
axes[1].set_yticks([])
axes[1].set_xlabel('Percentage of total orders')
axes[1].set_title('Proportion by fulfilment status', fontsize=13, fontweight='medium')
axes[1].legend(loc='lower center', bbox_to_anchor=(0.5, -0.35),
               ncol=2, fontsize=9, frameon=False)

fig.suptitle(f"{len(anomalies)/total*100:.2f}% of orders have a ship date before purchase date (anomalies)", 
                    fontsize=16, fontweight='medium', y=1.02)

save_figure(fig, '01_fulfilment_status_overview.png')
plt.show()

In [ ]:
# Chart 2: Distribution of anomaly severity
# Show how many anomalies are just slightly negative (e.g., -1 day) vs.

fig, ax = plt.subplots(figsize=(12, 5))

# Negative values only, to show severity distribution
anomaly_days = anomalies['FULFILMENT_DAYS'].abs()  # take absolute value to show how many days before purchase the ship date was, regardless of direction

ax.hist(
    anomaly_days, 
    bins=50, 
    color='#C00000',
    alpha=0.8, 
    edgecolor='white',
    linewidth=0.5
)

median_value = anomaly_days.median()
ax.axvline(median_value, color='#1f4e79', linestyle='--', linewidth=2,
           label=f'Median: {median_value:.0f} days')

ax.set_title('Distribution of Anomaly Severity\n(How many days early was the ship date?)', 
             fontsize=14, fontweight='medium')
ax.set_xlabel('Days Early (Absolute Value)')
ax.set_ylabel('Number of orders')
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))  # format x-axis with commas

save_figure(fig, '01_anomaly_severity_distribution.png')
plt.show()

In [ ]:
# Chart 3: Anomaly rate by product
# Horizontal bar chart, much easier to read than a vertical one when there are many products, and it allows for longer product names without overlap

fig, ax = plt.subplots(figsize=(12, 8))

product_plot = product_analysis.sort_values('anomaly_rate')

bars = ax.barh(
    product_plot.index, 
    product_plot['anomaly_rate'], 
    color='#2E75B6', 
    alpha=0.8
)

# Add anomaly rate labels on the bars
for bar, rate in zip(bars, product_plot['anomaly_rate']):
    ax.text(
        bar.get_width() + 0.2,  # position label slightly to the right of the bar
        bar.get_y() + bar.get_height() / 2,  # vertically center the label
        f'{rate:.1f}%', 
        ha='left', va='center', fontsize=9
    )
 
# Add a vertical line to show the overall anomaly rate across all products for context    
overall_rate = (len(anomalies) / total * 100)
ax.axvline(overall_rate, color='#C00000', linestyle='--', linewidth=2, 
           label=f'Overall Anomaly Rate: {overall_rate:.1f}%')

ax.set_title('Anomaly Rate by Product\n(% of orders with ship date before purchase date)', 
             fontsize=14, fontweight='medium')
ax.set_xlabel('Anomaly Rate (%)')
ax.legend()

save_figure(fig, '01_anomaly_rate_by_product.png')
plt.show()

In [ ]:
# Chart 4: Anomaly rate over time
# Line chart showing how the anomaly rate changes month by month, to identify any trends or spikes

fig, ax = plt.subplots(figsize=(14, 6))

# Convert period index to string for plotting
time_plot = time_analysis.copy()
time_plot.index = time_plot.index.astype(str)

ax.plot(
    time_plot.index, 
    time_plot['anomaly_rate'], 
    marker='o', 
    color='#2E75B6', 
    linewidth=2, 
    markersize=6,
    label='Anomaly rate (%)'
)

# Add a shaded area under the line to highlight the anomaly rate trend over time
ax.fill_between(
    time_plot.index, 
    time_plot['anomaly_rate'], 
    color='#2E75B6', 
    alpha=0.1
)

# Add a horizontal line to show the overall anomaly rate across the entire period for context
ax.axhline(overall_rate, color='#C00000', linestyle='--', linewidth=2,
           label=f'Overall Anomaly Rate: {overall_rate:.1f}%')

ax.set_title('Anomaly Rate Over Time - is this a trend or a one-off spike?', 
             fontsize=14, fontweight='medium')
ax.set_xlabel('Month')
ax.set_ylabel('Anomaly Rate (%)')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')
ax.legend()

save_figure(fig, '01_anomaly_rate_over_time.png')
plt.show()

In [ ]:
# Chart 5: Fulfilment days - normal vs anomalies
# Give full picture of fulfilment time dist.

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# Left: normal orders distribution
ax[0].hist(
    normal['FULFILMENT_DAYS'].clip(upper=30), # clip to 30 days to focus on the main distribution and avoid extreme outliers dominating the chart
    bins=30,
    color='#2E75B6',
    alpha=0.8,
    edgecolor='white',
)
ax[0].set_title('Normal orders - fulfilment time\n(capped at 30 days for visibility)', 
                fontsize=14, fontweight='medium')
ax[0].set_xlabel('Fulfilment days (purchase date - ship date)')
ax[0].set_ylabel('Number of orders')

# Right: anomalies distribution - how many days before purchase was the ship date, to show the severity of anomalies
ax[1].hist(
    anomalies['FULFILMENT_DAYS'].clip(lower=-100), # clip to -100 days to focus on the main distribution of anomalies and avoid extreme outliers dominating the chart
    bins=30, 
    color='#C00000', 
    alpha=0.8, 
    edgecolor='white'
)
ax[1].set_title('Anomalies - how many days before purchase was the ship date?', 
                fontsize=14, fontweight='medium')
ax[1].set_xlabel('Fulfilment days (purchase date - ship date)')
ax[1].set_ylabel('Number of orders')

fig.suptitle('Fulfilment time distribution: normal orders vs anomalies', 
             fontsize=16, fontweight='medium', y=1.02)

save_figure(fig, '01_fulfilment_time_distribution_comparison.png')
plt.show()

## Hypothesis Testing

In [ ]:
# HYPOTHESIS: The anomalies are caused by timezone offset errors.
#
# If the purchase timestamp was recorded in the customer's LOCAL timezone
# but stored in the database as if it were UTC, then a customer in a timezone
# far west of UTC (e.g. UTC-12) would have their purchase time shifted
# FORWARD by up to 12 hours.
#
# Meanwhile the ship timestamp might have been recorded correctly in UTC.
# This could result in the purchase appearing to happen AFTER the shipment
# by a matter of hours — producing a small negative fulfilment day value.
#
# KEY TEST: If timezone error is the cause, most anomalies should be
# concentrated at -1 day (just a few hours off), not -100 or -600 days.

print('Testing the timezone error hypothesis for anomalies')
print()
print('If the anomalies are caused by timezone offset errors, we would expect to see a large concentration of anomalies with fulfilment days around -1 day.')
print()

# Count how many anomalies fall into each fulfilment day value.
day_counts = anomalies['FULFILMENT_DAYS'].value_counts().sort_index()

# Separate the anomalies into "close" (within 10 days) and 
# "far" (more than 10 days) to see if the majority are close to -1 day, which would support the timezone error hypothesis.
close_anomalies = day_counts[day_counts.index >= -10]
far_anomalies = day_counts[day_counts.index < -10]

pct_close = len(anomalies[anomalies['FULFILMENT_DAYS'] >= -10]) / len(anomalies) * 100
pct_far = len(anomalies[anomalies['FULFILMENT_DAYS'] < -10]) / len(anomalies) * 100 

print(f'Anomalies within 10 days of purchase date: {len(anomalies[anomalies["FULFILMENT_DAYS"] >= -10]):,} ({pct_close:.1f}%)')
print(f'Anomalies more than 10 days from purchase date: {len(anomalies[anomalies["FULFILMENT_DAYS"] < -10]):,} ({pct_far:.1f}%)')
print()
print('Exact breakdown for -1 to -10 days:')
for day, count in close_anomalies.items():
    print(f'  {day:>4} days:  {count:>5,} orders  ({count/len(anomalies)*100:.1f}%)')

In [ ]:
print('VERDICT: Timezone hypothesis REJECTED.')
print(f'Only {pct_close:.1f}% of anomalies are within 10 days. The maximum')
print('timezone difference on Earth is 26 hours — it cannot explain a 79-day gap.')

In [ ]:
# SECONDARY HYPOTHESIS: Pre-order / early fulfilment logic
#
# Some e-commerce platforms allow merchants to ship items from a warehouse
# before a scheduled release date. The 'purchase' date might be when the
# customer placed the pre-order, and the 'ship' date when the warehouse
# dispatched it — but the purchase_ts was backdated to when the order
# was RECEIVED by the system, not when the customer clicked buy.
#
# KEY TEST: Do high-demand launch products (Nintendo Switch, PS5)
# have disproportionately more extreme anomalies?

print('Checking High-Demand Products for Pre-Order Pattern')
print()

high_demand = ['Nintendo Switch', 'Sony PlayStation 5 Bundle']

for product in high_demand:
    product_anomalies = anomalies[anomalies['PRODUCT_NAME'] == product]
    all_product = df[df['PRODUCT_NAME'] == product]
    
    if len(product_anomalies) > 0:
        print(f'{product}:')
        print(f'  Total orders:     {len(all_product):,}')
        print(f'  Anomalous orders: {len(product_anomalies):,} ({len(product_anomalies)/len(all_product)*100:.1f}%)')
        print(f'  Most extreme:     {product_anomalies["FULFILMENT_DAYS"].min()} days')
        print(f'  Median anomaly:   {product_anomalies["FULFILMENT_DAYS"].median():.0f} days')
        print()

In [ ]:
print('VERDICT: Pre-order hypothesis STRONGLY SUPPORTED.')
print('Median anomalies of 79 days (Switch) and 62 days (PS5) align precisely')
print('with typical gaming hardware pre-order windows of 2-5 months.')
print('The anomaly distribution is cleanly bounded between 0-150 days —')
print('exactly the range you would expect from pre-order fulfilment.')

## Findings and Recommendations

In [ ]:
print('=' * 65)
print('  NOTEBOOK 1 FINDINGS — TEMPORAL ANOMALY INVESTIGATION')
print('=' * 65)
print()
print('FINDING 1 — Scale')
print(f'  1,997 orders ({len(anomalies)/total*100:.1f}% of the dataset) have a ship date')
print( '  that precedes the purchase date. This is a systemic issue,')
print( '  not a one-off error.')
print()
print('FINDING 2 — Severity: anomalies are large, not minor')
print(f'  The median anomaly is {anomalies["FULFILMENT_DAYS"].median():.0f} days. 92.8% of anomalies')
print( '  are more than 10 days from the purchase date. The distribution')
print( '  is cleanly bounded between 0 and 150 days.')
print()
print('FINDING 3 — Platform: not platform-specific')
print( '  Website (9.2%) and mobile app (8.7%) have virtually identical')
print( '  anomaly rates. The issue is not caused by one platform.')
print()
print('FINDING 4 — Geography: three markets above average')
print( '  India (11.9%), Switzerland (12.2%), and South Korea (11.1%)')
print( '  are notably above the 9.1% overall average. This may reflect')
print( '  stronger pre-order cultures or different fulfilment practices')
print( '  in those markets.')
print()
print('FINDING 5 — Channel: social media is highest at 12.7%')
print( '  Social media anomaly rate (12.7%) is above the 8-9% seen in')
print( '  other channels. Social campaigns commonly drive pre-order')
print( '  interest for gaming hardware launches.')
print()
print('FINDING 6 — Time: persistent and consistent throughout dataset')
print( '  The anomaly rate is consistent across 2019-2020 with no sudden')
print( '  spikes. This confirms it is a structural recording practice,')
print( '  not a one-time system error.')
print()
print('FINDING 7 — Duplicate product name identified')
print( '  "27inches 4k gaming monitor" and "27in 4K gaming monitor" are')
print( '  the same product recorded under two different names. This is')
print( '  addressed formally in Notebook 3: Entity Resolution.')
print()
print('MOST LIKELY CAUSE: Pre-order fulfilment recording')
print( '  Timezone hypothesis was rejected — no timezone error can explain')
print( '  a 79-day gap. Pre-order hypothesis is strongly supported by the')
print( '  severity distribution, product-level data, and bounded 0-150 day range.')
print()
print('RECOMMENDATION — How to handle in downstream analysis')
print( '  Option A (Conservative): Exclude all 1,997 anomalous records.')
print( '  Option B (Pragmatic):    Flag with IS_ANOMALY=True and retain')
print( '                           for revenue/CLV analysis, but exclude')
print( '                           from any fulfilment time calculations.')
print()
print( '  We adopt Option B. These records represent real purchases and')
print( '  real revenue. Excluding them would under-count customers and')
print( '  distort CLV calculations in Project 02.')
print()
print('=' * 65)

## Export to Processed Data

In [ ]:
# Ensure the processed data directory exists for saving any cleaned datasets or outputs from this analysis

processed_data_path = project_root / 'data' / 'processed'
os.makedirs(processed_data_path, exist_ok=True)

output_1 = processed_data_path / 'orders_with_anomaly_flag.csv'
df.to_csv(output_1, index=False)

print(f'Output 1 saved: {output_1}')
print(f'Rows: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print()
print('Notebook 1 complete ✓')
print('Figures saved to reports/figures/')

In [ ]:
# Save the anomalous records to a CSV for reference
# This is useful for the Excel report and for auditing
import os
os.makedirs('../../data/processed', exist_ok=True)

anomalous_export = anomalies[[
    'USER_ID', 'ORDER_ID', 'PURCHASE_TS', 'SHIP_TS',
    'FULFILMENT_DAYS', 'PRODUCT_NAME', 'COUNTRY_CODE',
    'PURCHASE_PLATFORM', 'MARKETING_CHANNEL', 'USD_PRICE'
]].copy()

anomalous_export.to_csv('../../data/processed/anomalous_orders.csv', index=False)
print(f'Saved {len(anomalous_export):,} anomalous records → data/processed/anomalous_orders.csv')